In [2]:
# simulation and db connections
import os
os.environ['TANGOS_SIMULATION_FOLDER'] = '/myhome1/users/Bvogel4/data/Marvel/'
os.environ['TANGOS_DB_CONNECTION'] = '/myhome1/users/Bvogel4/data/db_backups/undergrads/Marvel.db'

print(os.listdir('plots'))

#os.environ['TANGOS_SIMULATION_FOLDER'] = '/myhome2/users/munshi/dwarf_volumes/'

['.ipynb_checkpoints', 'elektra.cosmo25cmb.4096g5HbwK1BH', 'rogue.cosmo25cmb.4096g5HbwK1BH', 'storm.cosmo25cmb.4096g5HbwK1BH']


In [2]:
# import modules + halo ids, define functions + constants
import tangos
import numpy as np
import matplotlib.pyplot as plt
import pynbody
import timeit
import pickle
import os

kpc3_cm3 = (3.086e63)          # conversion from kpc^3 to cm^3
m_H = (1.67e-27)/(1.989e30)    # mass of neutral H in Msol (mp in kg/Msol in kg)

conv = 1 / (kpc3_cm3 * m_H)

In [9]:
# storm simulation details 
sim_name = 'storm.cosmo25cmb.4096g5HbwK1BH'

# snapshot numbers 
snapshots = ['4096', '4032', '3936', '3840', '3744', '3648', '3552', '3456', '3360',
    '3264', '3168', '3072', '2976', '2880', '2784', '2688', '2592', '2496',
    '2400', '2304', '2208', '2112', '2016', '1920', '1824', '1728', '1632',
    '1536', '1440', '1344', '1248', '1152', '1056', '0960', '0864', '0768',
    '0672', '0576','0480','0384','0288']
print(len(snapshots))

storm_allIDs = np.array([[16.0, 20.0, 24.0, 25.0, 27.0, 29.0, 35.0, 38.0, 41.0, 47.0, 51.0, 70.0],
[16.0, 20.0, 24.0, 26.0, 27.0, 29.0, 35.0, 38.0, 41.0, 47.0, 50.0, 70.0],
[15.0, 19.0, 24.0, 26.0, 25.0, 30.0, 32.0, 36.0, 40.0, 44.0, 51.0, 73.0],
[16.0, 20.0, 25.0, 26.0, 24.0, 30.0, 34.0, 36.0, 41.0, 45.0, 50.0, 74.0],
[16.0, 20.0, 27.0, 26.0, 25.0, 31.0, 36.0, 38.0, 42.0, 48.0, 52.0, 75.0],
[17.0, 20.0, 29.0, 28.0, 26.0, 32.0, 36.0, 39.0, 41.0, 48.0, 53.0, 76.0],
[17.0, 20.0, 29.0, 28.0, 27.0, 35.0, 37.0, 40.0, 41.0, 49.0, 55.0, 77.0],
[17.0, 20.0, 29.0, 28.0, 27.0, 35.0, 36.0, 39.0, 41.0, 48.0, 54.0, 79.0],
[17.0, 20.0, 29.0, 28.0, 27.0, 35.0, 38.0, 39.0, 40.0, 48.0, 54.0, 78.0],
[17.0, 20.0, 29.0, 27.0, 28.0, 37.0, 39.0, 40.0, 41.0, 51.0, 58.0, 78.0],
[17.0, 20.0, 30.0, 27.0, 28.0, 41.0, 42.0, 43.0, 44.0, 55.0, 59.0, 79.0],
[17.0, 20.0, 31.0, 26.0, 28.0, 40.0, 41.0, 42.0, 44.0, 53.0, 56.0, 79.0],
[17.0, 21.0, 30.0, 28.0, 29.0, 41.0, 44.0, 45.0, 46.0, 55.0, 57.0, 80.0],
[17.0, 21.0, 32.0, 29.0, 30.0, 42.0, 44.0, 45.0, 46.0, 56.0, 57.0, 80.0],
[17.0, 22.0, 32.0, 30.0, 28.0, 41.0, 44.0, 42.0, 46.0, 57.0, 58.0, 80.0],
[18.0, 22.0, 30.0, 28.0, 27.0, 40.0, 46.0, 44.0, 45.0, 58.0, 56.0, 82.0],
[18.0, 22.0, 30.0, 31.0, 28.0, 39.0, 46.0, 42.0, 45.0, 62.0, 59.0, 82.0],
[20.0, 25.0, 30.0, 31.0, 28.0, 38.0, 47.0, 45.0, 46.0, 64.0, 59.0, 85.0],
[19.0, 28.0, 33.0, 34.0, 31.0, 39.0, 49.0, 48.0, 47.0, 66.0, 61.0, 88.0],
[22.0, 30.0, 34.0, 36.0, 33.0, 39.0, 47.0, 46.0, 48.0, 65.0, 60.0, 85.0],
[20.0, 30.0, 35.0, 37.0, 33.0, 39.0, 46.0, 47.0, 48.0, 63.0, 60.0, 85.0],
[21.0, 26.0, 35.0, 37.0, 34.0, 41.0, 45.0, 47.0, 48.0, 66.0, 63.0, 90.0],
[21.0, 25.0, 36.0, 37.0, 34.0, 43.0, 47.0, 48.0, 49.0, 63.0, 62.0, 87.0],
[21.0, 27.0, 38.0, 33.0, 36.0, 44.0, 48.0, 49.0, 50.0, 67.0, 61.0, 91.0],
[20.0, 30.0, 38.0, 29.0, 36.0, 49.0, 47.0, 48.0, 50.0, 67.0, 61.0, 89.0],
[25.0, 33.0, 37.0, 31.0, 32.0, 58.0, 47.0, 48.0, 50.0, 75.0, 62.0, 92.0],
[34.0, 33.0, 37.0, 31.0, 29.0, 62.0, 49.0, 46.0, 53.0, 97.0, 66.0, 94.0],
[40.0, 35.0, 42.0, 28.0, 31.0, 65.0, 53.0, 49.0, 58.0, 99.0, 68.0, 96.0],
[36.0, 33.0, 39.0, 45.0, 31.0, 64.0, 53.0, 51.0, 57.0, 104.0, 68.0, 99.0],
[39.0, 31.0, 36.0, 59.0, 35.0, 63.0, 51.0, 50.0, 54.0, 105.0, 69.0, 99.0],
[37.0, 29.0, 35.0, 55.0, 39.0, 63.0, 48.0, 54.0, 50.0, 98.0, 65.0, 94.0],
[36.0, 27.0, 38.0, 53.0, 42.0, 61.0, 47.0, 59.0, 51.0, 106.0, 67.0, 96.0],
[36.0, 30.0, 41.0, 61.0, 49.0, 67.0, 48.0, 64.0, 55.0, 113.0, 69.0, 96.0],
[37.0, 47.0, 50.0, 71.0, 56.0, 67.0, 51.0, 64.0, 55.0, 163.0, 73.0, 94.0],
[35.0, 44.0, 69.0, 111.0, 57.0, 83.0, 73.0, 63.0, 65.0, 203.0, 72.0, 107.0],
[37.0, 36.0, 83.0, 108.0, 56.0, 159.0, 75.0, 60.0, 68.0, 230.0, 69.0, 113.0],
[39.0, 40.0, 91.0, 116.0, 55.0, 181.0, 69.0, 66.0, 70.0, 249.0, 68.0, 115.0],
[38.0, 88.0, 83.0, 115.0, 51.0, 214.0, 75.0, 59.0, 56.0, 256.0, 67.0, 111.0],
[47.0, 128.0, 74.0, 103.0, 59.0, 160.0, 98.0, 183.0, 49.0, 258.0, 79.0, 107.0],
[44.0, 125.0, 74.0, 102.0, 57.0, 133.0, 89.0, 169.0, 43.0, 295.0, 86.0, 107.0],
[71.0, 81.0, 96.0, 104.0, 40.0, 496.0, 68.0, 149.0, 69.0, 242.0, 119.0, 77.0]])

print(len(storm_allIDs[:,0]))

# halo IDs at z=0
target_ids0 = [16, 20, 24, 25, 27, 29, 35, 38, 41, 47, 51, 70]

# if running only one snapshot at a time, use h = index
#h = 0

# if you want to run a bunch of snapshots use loop and tab EVERYTHING below 
for h in range(len(snapshots)):
        
    ts = f'00{snapshots[h]}'  # timestep
    
    # get sim
    start = timeit.default_timer()
    sim = pynbody.load(f'/myhome1/users/Bvogel4/data/Marvel/{sim_name}/{sim_name}.{ts}')
    sim.physical_units()
    stop = timeit.default_timer()
    print(f"sim {sim_name}.{ts} found in {(stop - start):.1f} sec")
    
    redshift = sim.properties['z']
    if h == 0: redshift = 0
    print(f"snapshot {ts} (z={redshift:.2f})")
    
    # get halos
    Start = timeit.default_timer()
    halos = sim.halos()
    stop = timeit.default_timer()
    print(f"halos found in {(stop - start):.1f} sec")
    
    # get IDs at this snapshot
    target_ids = storm_allIDs[h]
    
    # loop through all halos in snapshot
    for i in range(len(target_ids)):
        # if halo exists
        start = timeit.default_timer()
        try:
            # get halo i's ID list
            gas = halos[target_ids[i]].gas
            grp = gas['amiga.grp']
    
            # filter out free particles
            grp_filter = grp[grp != 0]
            if grp_filter.size == 0:
                print(f"ERROR: no bound particles in halo {target_ids[i]}")
                continue
                
            n_gas = len(gas)
            print(f" -----> Plotting halo {target_ids[i]} ({n_gas} particles)")
            rho = np.zeros(n_gas)
            temp = np.zeros(n_gas)
            
            rho = gas['rho'] *conv*(1e-1)
            temp = gas['temp']
    
            #what i chnaged------
            from scipy.stats import gaussian_kde
            xy = np.vstack([np.log10(rho), np.log10(temp)])
    
            z = gaussian_kde(xy)(xy)
    
            plt.figure(dpi=300)
            plt.scatter(rho, temp, c=z, s=10, cmap='viridis')
            plt.xscale('log')
            plt.yscale('log')
            plt.xlim(1e-6, 1e-1)
            plt.ylim(1e3, 1e5)
            # change sim name in title 
            plt.title(f'Storm halo {target_ids0[i]}: gas particles at z={redshift:.2f}')
            plt.xlabel(r"log($n_{\mathrm{H}}\ \mathrm{(cm^{-3})})$")
            plt.ylabel(r"$log(T\ \mathrm{(K)})$")
            plt.grid(True)
            
            filename = f"plots/{sim_name}/{sim_name}_halo{target_ids0[i]}/{sim_name}_halo{target_ids0[i]}_snap{ts}.png"
            plt.savefig(filename, dpi=300)
            plt.close()
    
            stop = timeit.default_timer()
            print(f"halo {target_ids0[i]} took {stop - start} seconds to run")
        except Exception as e:
            print(f'ERROR: {e} - skip')
            continue

41
41
sim storm.cosmo25cmb.4096g5HbwK1BH.004096 found in 0.1 sec
snapshot 004096 (z=0.00)


KeyboardInterrupt: 

In [5]:
# elektra simulation details 
sim_name = 'elektra.cosmo25cmb.4096g5HbwK1BH'

# snapshot numbers 
snapshots = ['4096', '4032', '3936', '3840', '3744', '3648', '3552', '3456', '3360',
    '3264', '3168', '3072', '2976', '2880', '2784', '2688', '2592', '2496',
    '2400', '2304', '2208', '2112', '2016', '1920', '1824', '1728', '1632',
    '1536', '1440', '1344', '1248', '1152', '1056', '0960', '0864', '0768',
    '0672', '0576','0480','0384','0288']
print(len(snapshots))

elektra_allIDs = np.array([[7, 13, 16, 19, 20, 21, 22, 24, 30, 40, 48, 62],
[7, 14, 17, 20, 21, 22, 23, 25, 31, 42, 47, 64],
[8, 14, 17, 19, 21, 22, 23, 25, 31, 42, 47, 64],
[10, 14, 17, 19, 20, 21, 22, 25, 31, 44, 48, 65],
[13, 14, 17, 19, 20, 21, 22, 25, 31, 44, 48, 63],
[15, 13, 17, 19, 20, 21, 22, 25, 32, 44, 49, 61],
[15, 13, 17, 20, 19, 21, 22, 25, 32, 45, 49, 58],
[15, 13, 17, 20, 19, 21, 22, 25, 32, 45, 49, 58],
[15, 13, 17, 20, 19, 21, 22, 25, 32, 43, 48, 55],
[15, 12, 18, 19, 20, 21, 22, 25, 32, 44, 48, 54],
[15, 13, 18, 19, 21, 20, 22, 25, 33, 46, 51, 55],
[13, 12, 18, 19, 23, 20, 22, 24, 33, 46, 51, 57],
[14, 12, 18, 19, 20, 22, 23, 25, 33, 47, 51, 57],
[13, 12, 17, 19, 18, 21, 22, 25, 33, 48, 53, 52],
[14, 13, 18, 20, 19, 23, 24, 26, 34, 51, 54, 50],
[15, 14, 18, 21, 19, 23, 25, 26, 34, 51, 54, 50],
[16, 14, 18, 21, 19, 23, 24, 25, 34, 51, 56, 50],
[14, 15, 19, 21, 20, 22, 25, 24, 35, 53, 56, 51],
[16, 17, 20, 22, 21, 23, 26, 25, 36, 54, 57, 52],
[16, 17, 20, 22, 21, 24, 27, 26, 38, 55, 58, 50],
[16, 17, 20, 22, 21, 25, 27, 26, 38, 55, 59, 51],
[17, 18, 21, 23, 22, 26, 27, 24, 37, 56, 60, 50],
[16, 17, 21, 23, 22, 25, 26, 24, 35, 58, 62, 50],
[18, 16, 21, 22, 20, 24, 30, 25, 37, 57, 66, 50],
[18, 16, 23, 22, 21, 24, 31, 32, 38, 62, 70, 52],
[18, 16, 21, 22, 23, 26, 39, 35, 37, 61, 71, 53],
[18, 16, 20, 22, 23, 26, 42, 39, 36, 62, 69, 55],
[19, 16, 20, 22, 23, 27, 43, 41, 37, 63, 67, 61],
[18, 17, 20, 25, 24, 27, 43, 40, 36, 62, 64, 60],
[14, 17, 20, 25, 26, 27, 45, 46, 36, 66, 67, 62],
[13, 20, 21, 23, 25, 27, 45, 46, 39, 67, 73, 65],
[16, 20, 23, 24, 25, 27, 42, 48, 47, 75, 81, 70],
[19, 20, 22, 24, 26, 28, 40, 42, 47, 75, 101, 87],
[21, 18, 24, 27, 25, 51, 42, 47, 57, 73, 117, 89],
[20, 18, 23, 28, 26, 87, 41, 61, 67, 103, 113, 98],
[17, 19, 28, 25, 23, 153, 38, 73, 47, 161, 101, 99],
[19, 25, 27, 26, 21, 138, 39, 87, 58, 152, 112, 102],
[19, 30, 44, 28, 21, 141, 40, 82, 153, 130, 114, 99],
[26, 37, 219, 21, 36, 260, 41, 49, 259, 143, 159, 86],
[27, 28, 196, 22, 55, 226, 30, 115, 261, 237, 161, 77],
[132, 79, 440, 98, 125, 291, 27, 150, 127, 228, 176, 54]])

print(len(elektra_allIDs[:,0]))
# halo IDs at z=0
target_ids0 = [7, 13, 16, 19, 20, 21, 22, 24, 30, 40, 48, 62]

# if running only one snapshot at a time, use h = index
#h = 36

# if you want to run a bunch of snapshots use loop and tab EVERYTHING below 
for h in range(39,len(snapshots)):
    
    ts = f'00{snapshots[h]}'  # timestep
    
    # get sim
    start = timeit.default_timer()
    sim = pynbody.load(f'/myhome1/users/Bvogel4/data/Marvel/{sim_name}/{sim_name}.{ts}')
    sim.physical_units()
    stop = timeit.default_timer()
    print(f"sim {sim_name}.{ts} found in {stop - start} sec")
    
    redshift = sim.properties['z']
    if h == 0: redshift = 0
    print(f"snapshot {ts} (z={redshift:.2f})")
    
    # get halos
    Start = timeit.default_timer()
    halos = sim.halos()
    stop = timeit.default_timer()
    print(f"halos found in {stop - start} sec")
    
    # get IDs at this snapshot
    target_ids = elektra_allIDs[h]
    
    # loop through all halos in snapshot
    for i in range(len(target_ids)):
        # if halo exists
        start = timeit.default_timer()
        try:
            # get halo i's ID list
            gas = halos[target_ids[i]].gas
            grp = gas['amiga.grp']
    
            # filter out free particles
            grp_filter = grp[grp != 0]
            if grp_filter.size == 0:
                print(f"ERROR: no bound particles in halo {target_ids[i]}")
                continue
                
            n_gas = len(gas)
            print(f"Plotting:{target_ids[i]} with {n_gas} particles")
            rho = np.zeros(n_gas)
            temp = np.zeros(n_gas)
            
            rho = gas['rho'] *conv*(1e-1)
            temp = gas['temp']
    
            #what i chnaged------
            from scipy.stats import gaussian_kde
            xy = np.vstack([np.log10(rho), np.log10(temp)])
    
            z = gaussian_kde(xy)(xy)
    
            plt.figure(dpi=300)
            plt.scatter(rho, temp, c=z, s=10, cmap='viridis')
            plt.xscale('log')
            plt.yscale('log')
            plt.xlim(1e-6, 1e-1)
            plt.ylim(1e3, 1e5)
            # change sim name in title 
            plt.title(f'Storm halo {target_ids0[i]}: gas particles at z={redshift:.2f}')
            plt.xlabel(r"log($n_{\mathrm{H}}\ \mathrm{(cm^{-3})})$")
            plt.ylabel(r"$log(T\ \mathrm{(K)})$")
            plt.grid(True)
            
            filename = f"plots/{sim_name}/{sim_name}_halo{target_ids0[i]}/{sim_name}_halo{target_ids0[i]}_snap{ts}.png"
            plt.savefig(filename, dpi=300)
            plt.close()
    
            stop = timeit.default_timer()
            print(f"halo {target_ids0[i]} took {stop - start} seconds to run")
        except Exception as e:
            print(f'ERROR: {e} - skip')
            continue

41
41
sim elektra.cosmo25cmb.4096g5HbwK1BH.000384 found in 1.8366041048429906 sec
snapshot 000384 (z=4.81)
halos found in 635.0757065671496 sec
Plotting:27 with 971 particles
halo 7 took 502.8473027450964 seconds to run
Plotting:28 with 891 particles
halo 13 took 10.701442118734121 seconds to run
ERROR: no bound particles in halo 196
Plotting:22 with 4681 particles
halo 19 took 19.756341030355543 seconds to run
Plotting:55 with 59 particles
halo 20 took 12.958656925708055 seconds to run
ERROR: no bound particles in halo 226
Plotting:30 with 1133 particles
halo 22 took 12.788749570026994 seconds to run
ERROR: no bound particles in halo 115
ERROR: no bound particles in halo 261
ERROR: no bound particles in halo 237
ERROR: no bound particles in halo 161
Plotting:77 with 24 particles
halo 62 took 9.400659881997854 seconds to run


OSError: File '/myhome1/users/Bvogel4/data/Marvel/elektra.cosmo25cmb.4096g5HbwK1BH/elektra.cosmo25cmb.4096g5HbwK1BH.000288': format not understood or does not exist

In [5]:
# rogue simulation details 
sim_name = 'rogue.cosmo25cmb.4096g5HbwK1BH'

# snapshot numbers 
snapshots = ['4096', '4032', '3936', '3840', '3744', '3648', '3552', '3456', '3360',
    '3264', '3168', '3072', '2976', '2880', '2784', '2688', '2592', '2496',
    '2400', '2304', '2208', '2112', '2016', '1920', '1824', '1728', '1632',
    '1536', '1440', '1344', '1248', '1152', '1056', '0960', '0864', '0768',
    '0672', '0576','0480','0384','0288']
print(len(snapshots))

rogue_allIDs = np.array([[24, 42],
[24, 42],
[24, 41],
[24, 40],
[24, 40],
[23, 40],
[23, 39],
[23, 38],
[25, 39],
[25, 37],
[24, 36],
[25, 39],
[24, 39],
[24, 40],
[25, 42],
[25, 45],
[26, 46],
[28, 47],
[27, 49],
[28, 47],
[97, 47],
[87, 47],
[84, 46],
[88, 46],
[90, 45],
[90, 45],
[91, 46],
[96, 48],
[96, 47],
[100, 51],
[108, 57],
[107, 61],
[108, 63],
[109, 65],
[111, 70],
[114, 69],
[100, 66],
[92, 71],
[315, 65],
[209, 58],
[238, 91]])

print(len(rogue_allIDs[:,0]))
# halo IDs at z=0
target_ids0 = [24,42]

# if running only one snapshot at a time, use h = index
#h = 0

# if you want to run a bunch of snapshots use loop and tab EVERYTHING below 
for h in range(len(snapshots)):
    
    ts = f'00{snapshots[h]}'  # timestep
    
    # get sim
    start = timeit.default_timer()
    sim = pynbody.load(f'/myhome1/users/Bvogel4/data/Marvel/{sim_name}/{sim_name}.{ts}')
    sim.physical_units()
    stop = timeit.default_timer()
    print(f"sim {sim_name}.{ts} found in {stop - start} sec")
    
    redshift = sim.properties['z']
    if h == 0: redshift = 0
    print(f"snapshot {ts} (z={redshift:.2f})")
    
    # get halos
    Start = timeit.default_timer()
    halos = sim.halos()
    stop = timeit.default_timer()
    print(f"halos found in {stop - start} sec")
    
    # get IDs at this snapshot
    target_ids = rogue_allIDs[h]
    
    # loop through all halos in snapshot
    for i in range(len(target_ids)):
        # if halo exists
        start = timeit.default_timer()
        try:
            # get halo i's ID list
            gas = halos[target_ids[i]].gas
            grp = gas['amiga.grp']
    
            # filter out free particles
            grp_filter = grp[grp != 0]
            if grp_filter.size == 0:
                print(f"ERROR: no bound particles in halo {target_ids[i]}")
                continue
                
            n_gas = len(gas)
            print(f"Plotting:{target_ids[i]} with {n_gas} particles")
            rho = np.zeros(n_gas)
            temp = np.zeros(n_gas)
            
            rho = gas['rho'] *conv*(1e-1)
            temp = gas['temp']
    
            #what i chnaged------
            from scipy.stats import gaussian_kde
            xy = np.vstack([np.log10(rho), np.log10(temp)])
    
            z = gaussian_kde(xy)(xy)
    
            plt.figure(dpi=300)
            plt.scatter(rho, temp, c=z, s=10, cmap='viridis')
            plt.xscale('log')
            plt.yscale('log')
            plt.xlim(1e-6, 1e-1)
            plt.ylim(1e3, 1e5)
            # change sim name in title 
            plt.title(f'Rogue halo {target_ids0[i]}: gas particles at z={redshift:.2f}')
            plt.xlabel(r"log($n_{\mathrm{H}}\ \mathrm{(cm^{-3})})$")
            plt.ylabel(r"$log(T\ \mathrm{(K)})$")
            plt.grid(True)
            
            filename = f"plots/{sim_name}/{sim_name}_halo{target_ids0[i]}/{sim_name}_halo{target_ids0[i]}_snap{ts}.png"
            plt.savefig(filename, dpi=300)
            plt.close()
    
            stop = timeit.default_timer()
            print(f"halo {target_ids0[i]} took {stop - start} seconds to run")
        except Exception as e:
            print(f'ERROR: {e} - skip')
            continue

41
41
sim rogue.cosmo25cmb.4096g5HbwK1BH.004096 found in 0.26143545005470514 sec
snapshot 004096 (z=0.00)
halos found in 119.1305949059315 sec
Plotting:24 with 5645 particles
halo 24 took 83.43694998323917 seconds to run
Plotting:42 with 59 particles
halo 42 took 2.9259250690229237 seconds to run
sim rogue.cosmo25cmb.4096g5HbwK1BH.004032 found in 1.136784311849624 sec
snapshot 004032 (z=0.02)
halos found in 119.25884388713166 sec
Plotting:24 with 5692 particles
halo 24 took 69.4659674782306 seconds to run
Plotting:42 with 42 particles
halo 42 took 2.1733446307480335 seconds to run
sim rogue.cosmo25cmb.4096g5HbwK1BH.003936 found in 2.2992064976133406 sec
snapshot 003936 (z=0.04)
halos found in 121.79014488868415 sec
Plotting:24 with 5746 particles
halo 24 took 82.82284821104258 seconds to run
Plotting:41 with 26 particles
halo 42 took 2.897323166951537 seconds to run
sim rogue.cosmo25cmb.4096g5HbwK1BH.003840 found in 0.9792248927988112 sec
snapshot 003840 (z=0.07)
halos found in 118.323

/myhome2/users/annaeng/anaconda3/lib/python3.8/site-packages/scipy/stats/kde.py:565: RuntimeWarning: Degrees of freedom <= 0 for slice
  self._data_covariance = atleast_2d(cov(self.dataset, rowvar=1,
/myhome2/users/annaeng/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:2680: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)
/myhome2/users/annaeng/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:2680: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


ERROR: array must not contain infs or NaNs - skip
Plotting:47 with 165 particles
halo 42 took 3.390073779039085 seconds to run
sim rogue.cosmo25cmb.4096g5HbwK1BH.002016 found in 1.2800865671597421 sec
snapshot 002016 (z=0.85)
halos found in 117.75320012494922 sec
ERROR: no bound particles in halo 84
Plotting:46 with 136 particles
halo 42 took 84.31444447627291 seconds to run
sim rogue.cosmo25cmb.4096g5HbwK1BH.001920 found in 0.26025418285280466 sec
snapshot 001920 (z=0.92)
halos found in 120.03054875321686 sec
Plotting:88 with 1 particles
ERROR: array must not contain infs or NaNs - skip
Plotting:46 with 115 particles
halo 42 took 3.0361219528131187 seconds to run
sim rogue.cosmo25cmb.4096g5HbwK1BH.001824 found in 3.9773754300549626 sec
snapshot 001824 (z=0.99)
halos found in 121.33273160317913 sec
Plotting:90 with 1 particles
ERROR: array must not contain infs or NaNs - skip
Plotting:45 with 96 particles
halo 42 took 2.726938286796212 seconds to run
sim rogue.cosmo25cmb.4096g5HbwK1BH.

In [7]:
# testing info: n halos,n particles, n gas, gas ids
print(f'n halos in sim: {len(halos)}')
halo0 = halos[7]
print(f'length all particles: {len(halo0["mass"])}')
gas = halo0.gas
ids = gas["amiga.grp"]
print(f'length gas particle array halo 0: {len(gas)}')
print(f'amiga grp ids for gas particles: {len(ids)}')

if np.any(ids) == 0:
    print('zero')
elif np.any(ids) != 7:
    print('not all 7')
else:
    print(ids[0])
gas.keys()
metals = gas['metals']
print(gas['amiga.grp'])
if np.any(metals) != 0:
    print('metals present')

NameError: name 'halos' is not defined